In [ ]:
from mne import SourceEstimate

import helper_functions as hf
import mne
import os.path as op
import numpy as np
from mne_denoise.zapline import ZapLine



In [ ]:
#load data

ss = hf.settings_dict()
subject = 1
raw_fname = op.join(ss['raw_dir'], ss['raw_path1_list'][subject])

raw = mne.io.read_raw_fif(raw_fname, preload=True)



In [ ]:
print(raw.info)

In [ ]:
#inspect for bad channels
# %matplotlib qt
raw.plot(block=True)

In [ ]:
#exclude bad channels (MEG1143)
raw_filtered = raw.copy() #.crop(tmin=0, tmax=160.5)
raw_filtered.info['bads'] = ss['bads']
raw_filtered.pick(picks=['grad', 'stim'])


del(raw)
#this was done in the original pre-processing
# raw.set_channel_types(mapping={'EMG001': 'eeg'})
# raw.set_channel_types(mapping={'EMG002': 'eeg'})
# raw.set_channel_types(mapping={'EOG003': 'eeg'})
# raw.set_channel_types(mapping={'EOG004': 'eeg'})
# raw.set_channel_types(mapping={'EMG005': 'eeg'})
# raw.set_channel_types(mapping={'MISC001': 'eeg'})

In [ ]:
#plot signal power for each frequency
raw_filtered.compute_psd(fmax=130).plot()

In [ ]:
#line noise removal
raw_filtered.notch_filter(freqs=[50], method='spectrum_fit')

In [ ]:
#plot signal power for each frequency
raw_filtered.compute_psd(fmax=130).plot()

In [ ]:
#filter 35 - 45 Hz


raw_filtered.filter(l_freq=35,h_freq=45)

print(raw_filtered.info)

In [ ]:
#plot signal power for each frequency
raw_filtered.compute_psd(fmax=130).plot()

In [ ]:
#cut data into epochs
tmin = -0.5
tmax = 4.5

events = mne.find_events(raw_filtered)
event_id_list =  ss['event_id_list']

epochs = mne.Epochs(raw_filtered, events, event_id_list, tmin=tmin, tmax=tmax, preload=True)

del(raw_filtered)



In [ ]:
#check that all trials are present
print(epochs.get_data().shape)
print(epochs.info)

In [ ]:
epochs.plot(picks='stim')

In [ ]:
#average
evokeds = {}

for event_id in event_id_list:
    evokeds[event_id] = epochs[event_id].average()


In [ ]:
from mne.beamformer import make_lcmv, apply_lcmv_epochs

#Source localization
evoked = evokeds[1]
evoked.pick('grad')

data_cov = mne.compute_covariance(epochs, tmin=0.01, tmax=tmax)

filter = make_lcmv(
    evoked.info,
    fwd,
    data_cov=data_cov,
    reg=0.05,
    pick_ori='max-power',
    weight_norm='nai'
)


stcs = apply_lcmv_epochs(evoked, filter)

In [ ]:
#Load forward model (later I should probably make it from scratch)
fwd_fname = op.join(ss['fwd_dir'], '0002_TCZ-fwd.fif')
fwd = mne.read_forward_solution(fwd_fname)


In [ ]:
stc = stcs[0].copy()
del stcs


In [ ]:
subject = "fs_0002_TCZ"            # or your subject ID
subjects_dir = "/home/elias/Documents/masters/Thesis/iliasgdk_thesis/scratch/fs_subjects_dir"

stc.copy().plot(src= fwd['src'],subject=subject,subjects_dir=subjects_dir)

In [ ]:
#apply hilbert transform
from scipy.signal import hilbert

analytic = hilbert(stc.data, axis=1)
envelope = np.abs(analytic)       # amplitude envelope
stc_power = stc.copy()
stc_power._data = envelope


In [ ]:
# =========================
# AVERAGE POWER OVER STEADY WINDOW
# =========================
trial_power = []

for stc in power_stcs:
    stc_crop = stc.copy().crop(tmin=0.5, tmax=4.0)
    trial_power.append(stc_crop.data.mean(axis=1))

mean_power = np.mean(trial_power, axis=0)
stc_mean = power_stcs[0].copy()
stc_mean._data = mean_power[:, np.newaxis]

# =========================
# BASELINE CONTRAST
# =========================
stc_example = power_stcs[0]

baseline = stc_example.copy().crop(tmin=-1, tmax=0)
active = stc_example.copy().crop(tmin=0.5, tmax=4)

baseline_power = baseline.data.mean(axis=1)
active_power = active.data.mean(axis=1)
relative = active_power / baseline_power

stc_contrast = stc_example.copy()
stc_contrast._data = relative[:, np.newaxis]

# =========================
# PLOT MEAN 40 Hz POWER
# =========================
print("Plotting mean 40 Hz source power...")
brain = stc_mean.plot(
    subject=subject,
    subjects_dir=subjects_dir,
    hemi="both",
    time_label="40 Hz Power",
    clim="auto"
)

# =========================
# PLOT BASELINE CONTRAST
# =========================
print("Plotting baseline contrast (active / baseline)...")
brain2 = stc_contrast.plot(
    subject=subject,
    subjects_dir=subjects_dir,
    hemi="both",
    time_label="Relative 40 Hz Power",
    clim="auto"
)